# Column Transformers and Pipelines

<img src = "./assets/lego.webp" width = "450">

[Image source](https://www.highsnobiety.com/p/lego-transformers-optimus-prime/)

In the [feature engineering basics notebook](01_feature_engineering_basics.ipynb) you fitted every transformer by hand, one column at a time. This notebook automates that work: a `ColumnTransformer` applies the right transformation to each column, a `Pipeline` chains steps one after another, and a single pipeline can hold all the preprocessing **and** the model.

## Learning Objectives

At the end of this notebook, you should be able to:

- Apply different transformations to different columns with `ColumnTransformer`.
- Chain several transformations on the same columns with `Pipeline`.
- Train a linear regression inside a pipeline that is fitted on the training data only.
- Extract the coefficients of the fitted model together with their feature names.

## Penguin Dataset

We will use the Palmer Penguins dataset, which records physical measurements of three penguin species.

### Business Goal
> Predict the body mass of a penguin from the input features `flipper_length_mm`, `bill_length_mm`, `species` and `sex`.

### Load Packages

Documentation: [`set_config`](https://scikit-learn.org/stable/modules/generated/sklearn.set_config.html).

In [ ]:
# data analysis stack
import pandas as pd

# data visualisation stack
import matplotlib.pyplot as plt
import seaborn as sns

# machine learning stack
from sklearn import set_config
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, RobustScaler, StandardScaler

sns.set_style("whitegrid")

# Make every scikit-learn transformer return a pandas DataFrame instead of a NumPy array
set_config(transform_output="pandas")

### Load Data

In [ ]:
df = pd.read_csv("./data/penguins.csv")
df.head()

### Features and Target

In [ ]:
numerical_features = ["flipper_length_mm", "bill_length_mm"]

categorical_features = ["species", "sex"]

features = numerical_features + categorical_features

target_variable = "body_mass_g"

### Feature-Target Separation

In [ ]:
# Feature matrix
X = df[features]

# Target column
y = df[target_variable]

### Train-Test Split

Documentation: [`train_test_split`](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.train_test_split.html).

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=88, shuffle=True, stratify=X["species"]
)

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)

## Exploratory Data Analysis

Before building any transformer, check the training data for issues:

- Which variables have missing values?
- Which variables are binary, categorical or metric?
- Do the categorical variables contain non-numeric values?
- Do the metric features vary on different scales?

The answers decide which transformation each column needs: missing values need imputation, non-numeric categories need encoding, and features on different scales need scaling.


In [ ]:
# Assuming X_train is a DataFrame and y_train is a Series
df_train = pd.concat([X_train, y_train], axis=1)

print("Combined train data shape:", df_train.shape)

In [ ]:
df_train.isna().sum()

---

## Feature Engineering

scikit-learn provides two tools, `ColumnTransformer` and `Pipeline`, that simplify and automate feature engineering. Instead of fitting each transformer by hand, you describe the transformations once and scikit-learn applies them in the right order.

### `ColumnTransformer`

[`ColumnTransformer`](https://scikit-learn.org/stable/modules/generated/sklearn.compose.ColumnTransformer.html) lets you specify which columns receive which transformation, and then combines the transformed columns back into a single dataset.

Parameters:

- `transformers`: a list of tuples `(name, transformer, columns)`.
- `remainder`: what to do with the columns that no transformer selects. Choose either `"drop"` (remove them) or `"passthrough"` (keep them unchanged).

> **Note:** `ColumnTransformer` applies all its transformers in parallel, not one after another. Each transformer receives the original column, so if a column is listed in more than one transformer, the output contains one transformed version per transformer.

#### Building Our First Transformer

Documentation: [`OneHotEncoder`](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.OneHotEncoder.html), [`RobustScaler`](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.RobustScaler.html).

In [ ]:
# define our transformers - name, method, target
transformers = [
    ("ohe", OneHotEncoder(drop="first", sparse_output=False), ["species", "sex"]),
    ("scaler", RobustScaler(), ["bill_length_mm", "flipper_length_mm"]),
]

In [ ]:
# now we instantiate our ColumnTransformer() object
column_transformer = ColumnTransformer(transformers, remainder="drop")
column_transformer

The data still has missing values in `sex` and `flipper_length_mm`. Adding a `SimpleImputer` to this `ColumnTransformer` would not solve the problem: because the transformers run in parallel, the output would contain an imputed copy of `sex` and `flipper_length_mm` alongside one-hot encoded and scaled versions that still contain missing values.

What you need instead is a way to apply transformations sequentially, so that the imputer fills the gaps first and the encoder or scaler then works on the complete column. This is what `Pipeline` does.

### `Pipeline`

[`Pipeline`](https://scikit-learn.org/stable/modules/generated/sklearn.pipeline.Pipeline.html) applies several transformers one after another on the same column(s): the output of each step becomes the input of the next.

Parameters:

- `steps`: a list of tuples `(name, transformer)`.

#### Building a Pipeline Inside the Transformer

Documentation: [`SimpleImputer`](https://scikit-learn.org/stable/modules/generated/sklearn.impute.SimpleImputer.html).

In [ ]:
# Let's define the steps to impute and transform sex
sex_steps = [
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("sex_ohe", OneHotEncoder(drop="first", sparse_output=False)),
]

In [ ]:
# Let's instantiate the sex pipeline
sex_pipeline = Pipeline(steps=sex_steps)
sex_pipeline

In [ ]:
# Let's define the steps to impute and transform flipper_length_mm
flipper_steps = [
    ("imputer", SimpleImputer(strategy="median")),
    ("flipper_scaler", RobustScaler()),
]

In [ ]:
# Let's instantiate the flipper pipeline
flipper_pipeline = Pipeline(steps=flipper_steps)
flipper_pipeline

In [ ]:
# Let's build a new transformer to include this pipeline
transformers_2 = [
    ("sex_pipeline", sex_pipeline, ["sex"]),
    ("ohe", OneHotEncoder(drop="first", sparse_output=False), ["species"]),
    ("flipper_pipeline", flipper_pipeline, ["flipper_length_mm"]),
    ("scaler", RobustScaler(), ["bill_length_mm"]),
]

column_transformer_2 = ColumnTransformer(transformers=transformers_2, remainder="drop")
column_transformer_2

#### Fitting and Transforming

In [ ]:
# Fit the column transformer object ONLY using train data
column_transformer_2.fit(X_train)

In [ ]:
X_train.isna().sum()

In [ ]:
# Transform the data
X_train_fe = column_transformer_2.transform(X_train)
X_train_fe

Each column has been transformed according to the strategy defined for it, and there are no missing values left: the imputers inside the pipelines filled the gaps before the encoder and the scaler ran.

## Model Building

### Nesting Pipelines

A `Pipeline` can also end with a model. The final technique in this notebook is nesting: the whole `ColumnTransformer` (which itself contains pipelines) becomes the first step of a new pipeline, and a `LinearRegression` becomes the second step. Calling `fit` on this pipeline fits the preprocessing and the model on the training data in one call, and calling `predict` or `score` applies the same fitted preprocessing to new data before the model sees it.

Documentation: [`LinearRegression`](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LinearRegression.html).

In [ ]:
# build a pipeline containing our complete transformer and then a linear regression model
model_steps = [
    ("feature_engineering", column_transformer_2),
    ("linear_regression", LinearRegression()),
]
linear_model = Pipeline(steps=model_steps)
linear_model

#### Train the Model

In [ ]:
linear_model.fit(X_train, y_train)

In [ ]:
training_score = linear_model.score(X_train, y_train)
print(f"training r2 score: {round(training_score, 6)}")

### Model Evaluation

#### Model Weights


In [ ]:
column_step = linear_model.steps[0][1]
column_step

In [ ]:
model_step = linear_model.steps[1][1]
model_step

In [ ]:
coef_model = pd.DataFrame(
    data=model_step.coef_.reshape(1, -1),
    columns=column_step.get_feature_names_out(),
    index=pd.Index(["weight"]),
)

coef_model["intercept"] = model_step.intercept_
coef_model

#### Model Prediction

In [ ]:
y_pred_test = linear_model.predict(X_test)
y_pred_test

#### Model Performance

In [ ]:
test_score = linear_model.score(X_test, y_test)
print(f"test r2 score: {round(test_score, 6)}")

The model explains 86.3% of the variance in body mass on the training data ($R^2$ 0.863) and 87.3% on the test data (0.873). The test score is not lower than the training score, so the model does not overfit: a linear model with five features has little room to memorise the training data.

The coefficients of the one-hot encoded columns read as differences in grams. For example, a Gentoo penguin weighs about 628 g more than an Adelie penguin (the dropped reference category) with the same flipper length, bill length and sex.

The coefficients of the two scaled columns are in grams per interquartile range rather than grams per millimetre, because `RobustScaler` divides each feature by its interquartile range.

> Applying the pipeline to the test data guarantees that the test data is treated in exactly the same way as the training data: the imputers, the encoders and the scalers reuse the values they learned from the training data (medians, most frequent categories, scaling statistics) and never learn from the test data.

### Try It: Swap the Scaler

Build the same model pipeline again, but with `StandardScaler` instead of `RobustScaler` for `flipper_length_mm` and `bill_length_mm`. Fit it on the training data and compare its test $R^2$ with the one above. Why does the scaler make so little difference here?

In [ ]:
# TODO: copy transformers_2, replacing both RobustScaler() with StandardScaler()
transformers_3 = ...

# TODO: build the ColumnTransformer and a Pipeline with a LinearRegression at the end
standard_model = ...

# TODO: fit on the training data and print the test R2
...

<details><summary>Show answer</summary>

```python
transformers_3 = [
    ("sex_pipeline", sex_pipeline, ["sex"]),
    ("ohe", OneHotEncoder(drop="first", sparse_output=False), ["species"]),
    (
        "flipper_pipeline",
        Pipeline([("imputer", SimpleImputer(strategy="median")), ("flipper_scaler", StandardScaler())]),
        ["flipper_length_mm"],
    ),
    ("scaler", StandardScaler(), ["bill_length_mm"]),
]

standard_model = Pipeline(
    [
        ("feature_engineering", ColumnTransformer(transformers_3, remainder="drop")),
        ("linear_regression", LinearRegression()),
    ]
)
standard_model.fit(X_train, y_train)
print(f"test r2 score: {round(standard_model.score(X_test, y_test), 6)}")
```

The test $R^2$ is exactly the same: 0.872736. Plain linear regression is not affected by scaling: rescaling a feature only rescales its coefficient, and the predictions stay identical. Scaling matters for models that penalise the size of the coefficients, such as Ridge and Lasso in W3D2, or that compare distances between data points.

</details>

---

## Summary

In this notebook you:

- Applied different transformations to different columns with `ColumnTransformer`: encoding for the categories, scaling for the numbers.
- Chained imputation and encoding (or scaling) on the same column with `Pipeline`.
- Trained a linear regression inside one pipeline, fitted on the training data only, and scored it on the test data with the same preprocessing.
- Extracted the model's coefficients together with the feature names created by the transformers.

A pipeline makes the rule "fit on the training data, only transform the test data" automatic, which is exactly what cross-validation in W3D4 relies on.

## References & Further Reading

- [**Pipelines and composite estimators (scikit-learn)**](https://scikit-learn.org/stable/modules/compose.html): the user guide for `Pipeline` and `ColumnTransformer`.
- [**Column Transformer with Mixed Types (scikit-learn)**](https://scikit-learn.org/stable/auto_examples/compose/plot_column_transformer_mixed_types.html): a complete example with numeric and categorical features.